In [2]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = (SparkSession.builder.appName("Bus Arrival Prediction").config("spark.sql.shuffle.partitions", "4").getOrCreate())

In [7]:
import os
gtfs = {}
for file in os.listdir("Data/Raw"):
    if file.endswith(".txt"):
        name = file.replace(".txt", "")
        gtfs[name] = spark.read.csv(
            f"Data/Raw/{file}",
            header=True,
            inferSchema=True)
print(gtfs.keys())
vehicle = pd.read_csv("Data/Raw/vehicle.csv")
disruption = pd.read_csv("Data/Raw/distrubtion.csv")

dict_keys(['agency', 'calendar', 'calendar_dates', 'feed_info', 'routes', 'shapes', 'stops', 'stop_times', 'trips'])


# Cleaning Dataset

In [8]:
vehicle["RecordedTime"] = pd.to_datetime(vehicle["RecordedTime"])
vehicle["Latitude"] = pd.to_numeric(vehicle["Latitude"], errors="coerce")
vehicle["Longitude"] = pd.to_numeric(vehicle["Longitude"], errors="coerce")
vehicle = vehicle.drop_duplicates()
vehicle = vehicle.dropna(subset=["LineRef", "Latitude", "Longitude"])
vehicle.head()

,RecordedTime,LineRef,Operator,VehicleRef,Latitude,Longitude
0,2026-07-02 15:10:13+00:00,96A,A2BR,3303,51.981412,-0.233722
1,2026-07-02 17:33:39+00:00,16A,A2BR,3311,52.073968,0.008972
2,2026-07-02 18:12:15+00:00,811,A2BV,A2BV-24,53.368135,-3.069226
3,2026-07-02 19:16:10+00:00,1A,A2BV,A2BV-RE24_TDZ,53.365285,-3.065045
4,2026-07-02 17:11:17+00:00,106,A2BV,A2BV-YJ10_MHK,53.419548,-3.046595


In [10]:
# Convert datetime columns
disruption["CreationTime"] = pd.to_datetime(
    disruption["CreationTime"],
    format="mixed",
    utc=True
)

disruption["VersionedAtTime"] = pd.to_datetime(
    disruption["VersionedAtTime"],
    format="mixed",
    utc=True
)

# Remove duplicates
disruption = disruption.drop_duplicates()

# Fill missing values
disruption["MiscellaneousReason"] = disruption["MiscellaneousReason"].fillna("Unknown")
disruption["EquipmentReason"] = disruption["EquipmentReason"].fillna("Unknown")
disruption["InfoLinks"] = disruption["InfoLinks"].fillna("None")

print("Disruption Shape:", disruption.shape)

Disruption Shape: (374, 16)


In [11]:
print("\nVehicle Info")
vehicle.info()

print("\nDisruption Info")
disruption.info()

print("\nVehicle Sample")
print(vehicle.head())

print("\nDisruption Sample")
print(disruption.head())


Vehicle Info
<class 'pandas.core.frame.DataFrame'>
Index: 27488 entries, 0 to 27501
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   RecordedTime  27488 non-null  datetime64[ns, UTC]
 1   LineRef       27488 non-null  object             
 2   Operator      27488 non-null  object             
 3   VehicleRef    27488 non-null  object             
 4   Latitude      27488 non-null  float64            
 5   Longitude     27488 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(2), object(3)
memory usage: 1.5+ MB

Disruption Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   CreationTime         374 non-null    datetime64[ns, UTC]
 1   ParticipantRef       374 non-null    object           

In [12]:
# Remove newline characters and extra spaces
disruption = disruption.replace(r'^\s*$', pd.NA, regex=True)

# Fill missing values
disruption["Source"] = disruption["Source"].fillna("Unknown")
disruption["ValidityPeriod"] = disruption["ValidityPeriod"].fillna("Unknown")
disruption["PublicationWindow"] = disruption["PublicationWindow"].fillna("Unknown")
disruption["Consequences"] = disruption["Consequences"].fillna("Unknown")
disruption["InfoLinks"] = disruption["InfoLinks"].fillna("None")

In [13]:
disruption.isnull().sum()

CreationTime           0
ParticipantRef         0
SituationNumber        0
Version                0
Source                 0
VersionedAtTime        0
Progress               0
ValidityPeriod         0
PublicationWindow      0
MiscellaneousReason    0
Planned                0
Summary                0
Description            0
Consequences           0
InfoLinks              0
EquipmentReason        0
dtype: int64

# Feature Enginerring

In [14]:
gtfs["agency"] = gtfs["agency"].select("agency_id","agency_name","agency_noc")
gtfs["routes"] = gtfs["routes"].select("route_id","agency_id","route_short_name","route_long_name","route_type")
gtfs["trips"] = gtfs["trips"].select("trip_id","route_id","service_id","trip_headsign","direction_id","shape_id","wheelchair_accessible")
gtfs["stop_times"] = gtfs["stop_times"].select(
    "trip_id",
    "arrival_time",
    "departure_time",
    "stop_id",
    "stop_sequence",
    "pickup_type",
    "drop_off_type",
    "timepoint"
)
gtfs["stops"] = gtfs["stops"].select("stop_id","stop_name","stop_lat","stop_lon","wheelchair_boarding")
gtfs["calendar"] = gtfs["calendar"].select("service_id","monday","tuesday","wednesday","thursday","friday","saturday","sunday","start_date","end_date")
vehicle = vehicle[[
    "RecordedTime",
    "LineRef",
    "Operator",
    "VehicleRef",
    "Latitude",
    "Longitude"
]]

In [15]:
disruption = disruption[[
    "CreationTime",
    "ParticipantRef",
    "Progress",
    "MiscellaneousReason",
    "Planned",
    "Summary",
    "Description",
    "EquipmentReason"
]]

In [16]:
for name, df in gtfs.items():
    print(f"\n{name.upper()}")
    print(df.columns)


AGENCY
['agency_id', 'agency_name', 'agency_noc']

CALENDAR
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

CALENDAR_DATES
['service_id', 'date', 'exception_type']

FEED_INFO
['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version']

ROUTES
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']

SHAPES
['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled']

STOPS
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'wheelchair_boarding']

STOP_TIMES
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'timepoint']

TRIPS
['trip_id', 'route_id', 'service_id', 'trip_headsign', 'direction_id', 'shape_id', 'wheelchair_accessible']


In [17]:
print("\nVehicle Columns")
print(vehicle.columns.tolist())

print("\nDisruption Columns")
print(disruption.columns.tolist())


Vehicle Columns
['RecordedTime', 'LineRef', 'Operator', 'VehicleRef', 'Latitude', 'Longitude']

Disruption Columns
['CreationTime', 'ParticipantRef', 'Progress', 'MiscellaneousReason', 'Planned', 'Summary', 'Description', 'EquipmentReason']


In [ ]:
a